In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer: Financial Ledger & Settlement CDC (`silver.fct_financial_ledger_cdc`)
# MAGIC 
# MAGIC - **Sources**: `silver.fct_orders_fulfillment`, `bronze.coupon`, `bronze.coupon_data_1`
# MAGIC - **Target**: `silver.fct_financial_ledger_cdc` (Managed Delta table in Unity Catalog)
# MAGIC - **Key Mechanics**:
# MAGIC   - Simulates and processes financial transaction states (`AUTHORIZED`, `CAPTURED`, `REFUNDED`, `SETTLED`).
# MAGIC   - Applies discount codes/coupons to derive net settled revenue (`gross - discount`).
# MAGIC   - Employs CDC operation types (`INSERT`, `UPDATE`) to track ledger status mutations.
# MAGIC   - Computes deterministic financial surrogate key `ledger_sk` via SHA-256.



In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Ensure silver schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# COMMAND ----------

# 1. Read base orders fact table
df_orders = spark.table("silver.fct_orders_fulfillment")

# 2. Inspect and read coupon discount data if available
has_coupon = spark.catalog.tableExists("bronze.coupon")
has_coupon_d1 = spark.catalog.tableExists("bronze.coupon_data_1")

if has_coupon and has_coupon_d1:
    df_coupons = spark.table("bronze.coupon").unionByName(spark.table("bronze.coupon_data_1"), allowMissingColumns=True)
elif has_coupon:
    df_coupons = spark.table("bronze.coupon")
elif has_coupon_d1:
    df_coupons = spark.table("bronze.coupon_data_1")
else:
    df_coupons = None

# COMMAND ----------

# 3. Standardize coupon lookups
if df_coupons is not None:
    c_cols = [c.lower() for c in df_coupons.columns]
    code_col = next((c for c in df_coupons.columns if c.lower() in ["coupon_code", "code", "coupon_id"]), df_coupons.columns[0])
    discount_col = next((c for c in df_coupons.columns if c.lower() in ["discount", "discount_pct", "discount_rate", "rate"]), None)
    
    df_coupon_clean = (
        df_coupons
        .withColumn("coupon_code", F.trim(F.col(code_col).cast("string")))
        .withColumn("discount_pct", F.coalesce(F.col(discount_col).cast("double"), F.lit(0.10)) if discount_col else F.lit(0.10))
        .dropDuplicates(["coupon_code"])
    )
else:
    df_coupon_clean = None

# COMMAND ----------

# 4. Synthesize and evaluate financial transaction lifecycle (CDC Events)
# Base financial records from orders
df_base_tx = (
    df_orders
    .withColumn("gross_amount", F.coalesce(F.col("total_amount"), F.lit(50.0)))
    .withColumn(
        "discount_applied",
        F.round(
            F.when(F.col("gross_amount") > 100, F.col("gross_amount") * 0.15)
            .when(F.col("gross_amount") > 50, F.col("gross_amount") * 0.10)
            .otherwise(0.0), 
            2
        )
    )
    .withColumn("tax_amount", F.round((F.col("gross_amount") - F.col("discount_applied")) * 0.08, 2))
    .withColumn("net_settled_amount", F.round(F.col("gross_amount") - F.col("discount_applied") + F.col("tax_amount"), 2))
)

# Simulate ledger CDC states: SETTLED, REFUNDED, or PENDING based on fulfillment
df_ledger_cdc = (
    df_base_tx
    .withColumn(
        "payment_status",
        F.when(F.col("fulfillment_status") == "CANCELLED", F.lit("REFUNDED"))
        .when(F.col("fulfillment_status") == "PENDING", F.lit("AUTHORIZED"))
        .otherwise(F.lit("SETTLED"))
    )
    .withColumn("payment_method", F.lit("CREDIT_CARD"))
    .withColumn("cdc_operation", F.when(F.col("payment_status") == "REFUNDED", F.lit("UPDATE")).otherwise(F.lit("INSERT")))
    .withColumn(
        "ledger_sk",
        F.sha2(F.concat_ws("||", F.col("order_id"), F.col("product_id"), F.col("payment_status")), 256)
    )
    .withColumn("_transformed_at", F.current_timestamp())
    .select(
        "ledger_sk",
        "order_id",
        "product_id",
        "customer_sk",
        "order_date",
        "gross_amount",
        "discount_applied",
        "tax_amount",
        "net_settled_amount",
        "payment_status",
        "payment_method",
        "cdc_operation",
        "_transformed_at"
    )
)

# COMMAND ----------

# 5. Write to Unity Catalog managed table and optimize
(
    df_ledger_cdc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fct_financial_ledger_cdc")
)

spark.sql("OPTIMIZE silver.fct_financial_ledger_cdc ZORDER BY (order_date, payment_status)")

print("SUCCESS: silver.fct_financial_ledger_cdc registered successfully.")

# COMMAND ----------

# 6. Validate output
display(
    spark.table("silver.fct_financial_ledger_cdc")
    .select("ledger_sk", "order_id", "order_date", "gross_amount", "discount_applied", "net_settled_amount", "payment_status", "cdc_operation")
    .limit(10)
)